# BTS Digital Twin (NVS) — Error-guided refine (thử nghiệm)

**Ý tưởng (đề xuất của user, 2026-07-18):** nạp lại 1 checkpoint ĐÃ TRAIN (chỉ cần
`point_cloud.ply`, không cần `.pth`), render lại chính các pose ẢNH TRAIN, so với ảnh
GT thật (có sẵn 100%, khác test/holdout không có GT) để tìm vùng pixel còn lỗi cao,
rồi train tiếp 1 đợt NGẮN ưu tiên loss vào đúng vùng đó.

**CHƯA test trên GPU thật** — mọi logic đã verify kỹ bằng cách đọc trực tiếp source
thật của `gaussian-splatting` (commit đã pin) + test cục bộ (patch áp sạch, công thức
weight, round-trip file 16-bit PNG) — nhưng chưa có lần chạy GPU thật nào xác nhận đợt
tinh chỉnh này THỰC SỰ cải thiện Score. Notebook này tự đo Score TRƯỚC/SAU để biết ngay.

**Giả định kỹ thuật quan trọng cần biết:** mask lỗi chỉ đo được trên ảnh TRAIN (có GT).
Giả định ngầm là vùng model tái tạo kém trên train cũng kém tương tự ở pose lân cận
trong holdout/test — CHƯA có bằng chứng thực nghiệm xác nhận. Đây chính là lý do phải
tự đo Score holdout trước/sau, không tin bằng trực giác.

Yêu cầu: đã có 1 checkpoint (`gs_model/` — thư mục, không phải chỉ file `.ply`) từ 1
lần chạy `kaggle_private.ipynb` trước đó, đã tải lên Google Drive (thư mục, share
"Anyone with the link").

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Clone SẠCH (chưa vá antenna-focus hay error-refine nào) — patch error-refine áp ở
Bước 6, không dùng chung được với antenna-focus (xem docstring
`apply_error_refine_patch.py`).

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `Hướng đi.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"
GIT_BRANCH = "coordination/round1-status"  # <-- nhánh chứa toàn bộ hạ tầng Round 2 (holdout eval, port kỹ thuật, TRR) — ĐỔI nếu đã merge vào main

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA_ROUND2/<scene>/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA_ROUND2`
cũng được — cell dưới tự dò tìm thư mục `VAI_NVS_DATA_ROUND2` ở bất kỳ độ sâu nào
trong zip).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục VAI_NVS_DATA_ROUND2 ...")

In [ ]:
# Tự dò thư mục chứa các scene round 2 phẳng (HCM0421/, chair/, bonsai/...) ở bất
# kỳ đâu trong zip vừa giải nén, rồi symlink về đúng vị trí mà
# pipeline/common/scenes.py cần: /kaggle/working/Dataset/VAI_NVS_DATA_ROUND2
#
# KHÔNG bắt buộc thư mục bọc ngoài phải tên đúng "VAI_NVS_DATA_ROUND2" — chỉ cần
# TÌM ĐƯỢC 1 thư mục (kể cả chính gốc giải nén, nếu zip không có lớp bọc ngoài)
# chứa đủ NHIỀU scene mong đợi trực tiếp bên trong. Bản cũ bắt buộc đúng tên thư
# mục nên sẽ báo lỗi "Không tìm thấy..." nếu file zip của bạn giải nén ra không có
# đúng lớp thư mục tên "VAI_NVS_DATA_ROUND2" đó (vd giải nén thẳng ra HCM0421/ ở
# gốc, hoặc thư mục bọc ngoài đặt tên khác) — dù dữ liệu vẫn đầy đủ.
#
# Danh sách tên scene lặp lại thủ công ở đây (không import common.scenes) vì
# sys.path chưa trỏ tới pipeline/ ở bước này (việc đó làm ở cell kiểm tra ngay
# sau) — giữ đồng bộ với BTS_SCENES/GENERIC_SCENES trong pipeline/common/scenes.py
# nếu sau này thêm/bớt scene.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4  # đủ scene trùng khớp để tin đây đúng là thư mục dataset (tránh khớp nhầm thư mục rác)

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (HCM0421/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/..." trước bản thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break  # khớp đủ cả 7 — dừng sớm, khỏi walk tiếp cho nhanh

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        "Không tìm thấy thư mục nào chứa đủ scene round 2 (HCM0421, chair, bonsai...) "
        f"trong zip vừa giải nén (khớp nhiều nhất: {best_match}/7, cần >= {_MIN_MATCH}).\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — chạy `!find {RAW_ROOT} -maxdepth 3` ở 1 cell "
        "khác để xem cấu trúc thật, đối chiếu lại với file zip đã upload lên Google Drive."
    )

print(f"Tìm thấy ({best_match}/7 scene khớp):", found)
target = Path("/kaggle/working/Dataset/VAI_NVS_DATA_ROUND2")
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 7 scene round 2 + scene nào có sparse hợp lệ — dataset
# đầy đủ thì kỳ vọng has_valid_provided_sparse=True cho CẢ 7 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa VAI_NVS_DATA_ROUND2 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.domain:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 — Cấu hình scene + checkpoint có sẵn

- `SCENE`: tên scene của checkpoint đã có (vd `HCM0421`).
- `CHECKPOINT_DRIVE_LINK`: link Drive tới THƯ MỤC `gs_model` (không phải chỉ file
  `.ply`) — checkpoint gốc PHẢI train ở `MODE="holdout"` (cần holdout GT để tự đo Score
  trước/sau, xem Bước 9) và PHẢI **KHÔNG** dùng `ANTENNA_FOCUS=1` (mask lỗi + antenna
  patch không tương thích nhau ở 1 checkpoint, xem docstring `apply_error_refine_patch.py`).
- `REFINE_ITERATIONS`: ngân sách tinh chỉnh THÊM (không phải train lại từ đầu) — mặc
  định 3000, nhỏ hơn nhiều so với 15000/30000 lúc train gốc.

In [ ]:
SCENE = "HCM0421"
CHECKPOINT_DRIVE_LINK = ""  # <-- dán link Drive tới thư mục gs_model (không phải file .ply đơn)
REFINE_ITERATIONS = 3000    # <-- ngân sách tinh chỉnh thêm (không phải train lại từ đầu)
MAX_ERROR_WEIGHT = 6.0      # <-- trọng số loss tối đa ở vùng lỗi cao nhất (xem 12_generate_error_mask.py)

assert CHECKPOINT_DRIVE_LINK, "Chưa điền CHECKPOINT_DRIVE_LINK — dán link Drive tới thư mục gs_model."
print(f"SCENE={SCENE}  REFINE_ITERATIONS={REFINE_ITERATIONS}  MAX_ERROR_WEIGHT={MAX_ERROR_WEIGHT}")

## Bước 6 — Tải checkpoint có sẵn từ Google Drive

In [ ]:
import shutil
from pathlib import Path

dest_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}")
dest_dir.mkdir(parents=True, exist_ok=True)
gs_model_dst = dest_dir / "gs_model"
shutil.rmtree(gs_model_dst, ignore_errors=True)

raw_dl_dir = Path(f"/kaggle/working/_ckpt_raw/{SCENE}")
shutil.rmtree(raw_dl_dir, ignore_errors=True)
raw_dl_dir.mkdir(parents=True, exist_ok=True)
print(f"===== {SCENE}: tải thư mục gs_model từ Drive =====")
!gdown --fuzzy --folder "{CHECKPOINT_DRIVE_LINK}" -O "{raw_dl_dir}"

# Cùng logic đã verify ở kaggle_submission.ipynb — gdown --folder có thể tự thêm 1 lớp
# thư mục con, tự dò lớp chứa "cfg_args" thay vì giả định cứng độ sâu.
candidates = [p.parent for p in raw_dl_dir.rglob("cfg_args")]
assert candidates, (
    f"Tải xong nhưng KHÔNG tìm thấy file 'cfg_args' trong {raw_dl_dir} — kiểm tra lại link Drive "
    f"có đúng là THƯ MỤC gs_model/ (không phải chỉ mỗi point_cloud.ply) và đã share "
    f"\"Anyone with the link\" chưa.")
src_root = candidates[0]
assert (src_root / "pipeline_train_flags.json").exists(), (
    f"Thiếu pipeline_train_flags.json trong {src_root} — thư mục gs_model tải lên Drive phải "
    f"nguyên vẹn (không tự xoá bớt file con nào).")

shutil.copytree(src_root, gs_model_dst)
shutil.rmtree(raw_dl_dir, ignore_errors=True)

import json as _json
from argparse import Namespace as _Namespace
_cfg = eval((gs_model_dst / "cfg_args").read_text(), {"Namespace": _Namespace})
_flags = _json.loads((gs_model_dst / "pipeline_train_flags.json").read_text())
assert not _flags.get("antenna_focus", False), (
    "Checkpoint này train với ANTENNA_FOCUS=1 — KHÔNG tương thích với error-refine "
    "(xem docstring apply_error_refine_patch.py). Dùng checkpoint khác (ANTENNA_FOCUS=0).")
CKPT_SH_DEGREE = vars(_cfg).get("sh_degree", 3)
CKPT_ANTIALIASING = bool(_flags.get("antialiasing", False))
CKPT_ITERATION = max(int(p.name.split("_")[-1]) for p in (gs_model_dst / "point_cloud").glob("iteration_*"))
print(f"-> OK, {gs_model_dst}")
print(f"-> Đọc được: sh_degree={CKPT_SH_DEGREE}  antialiasing={CKPT_ANTIALIASING}  "
      f"iteration cao nhất có sẵn={CKPT_ITERATION}")

## Bước 7 — Tái tạo `colmap/dense/images/` (bắt buộc chính xác pixel-for-pixel)

`03_train_3dgs.sh` đã tự xoá thư mục này sau khi train xong (dọn đĩa bình thường) —
mask lỗi cần ảnh ĐÃ undistort chính xác (không phải bản xấp xỉ resize như
`10_sanity_check_render.py` dùng cho việc kiểm tra nhanh), nên phải chạy lại COLMAP
undistort. Deterministic (seed=42 cố định) nên ra lại đúng holdout split/undistort như
lúc train gốc.

In [ ]:
holdout_dir = f"/kaggle/working/pipeline/work/{SCENE}/holdout"
import os
if not os.path.isdir(holdout_dir):
    !python /kaggle/working/pipeline/scripts/00_make_holdout_split.py --scene {SCENE}
else:
    print(f"Đã có {holdout_dir} — bỏ qua tạo lại.")
!python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE} --holdout

## Bước 8 — Đo Score TRƯỚC khi tinh chỉnh (baseline, để so sánh)

Render + chấm điểm holdout bằng ĐÚNG checkpoint gốc — nếu score này khác nhiều so với
lần train gốc (vd lệch antialiasing tự phát hiện sai), DỪNG LẠI kiểm tra trước khi đi
tiếp.

In [ ]:
!python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {SCENE} \
    --poses_csv /kaggle/working/pipeline/work/{SCENE}/holdout/holdout_poses.csv \
    --out_dir /kaggle/working/pipeline/work/{SCENE}/holdout_renders \
    --iteration {CKPT_ITERATION}
!python /kaggle/working/pipeline/scripts/05_eval_metrics.py --scene {SCENE}

import shutil
shutil.copy(
    f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics.txt",
    f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics_BEFORE_refine.txt",
)
print("-> Đã lưu bản sao eval_metrics_BEFORE_refine.txt để so sánh sau Bước 11.")

## Bước 9 — Sinh error mask

Render lại chính pose TRAIN, so GT thật, sinh mask 16-bit PNG cho vùng lỗi cao.

In [ ]:
!python /kaggle/working/pipeline/scripts/12_generate_error_mask.py --scene {SCENE} \
    --iteration {CKPT_ITERATION} --max_weight {MAX_ERROR_WEIGHT}

## Bước 10 — Vá `GS_REPO` + train tiếp (error-guided refine)

`--densify_until_iter 0`: KHÔNG sinh thêm Gaussian mới, chỉ tối ưu lại vị trí/màu/
opacity Gaussian đã có — đúng tinh thần "tinh chỉnh", không phải train lại.

In [ ]:
!python /kaggle/working/pipeline/scripts/apply_error_refine_patch.py --gs_repo {os.environ['GS_REPO']}

GS_REPO_TRAIN = f"{os.environ['GS_REPO']}/train.py"
SOURCE_DIR = f"/kaggle/working/pipeline/work/{SCENE}/colmap/dense"
MODEL_DIR = f"/kaggle/working/pipeline/work/{SCENE}/gs_model"
ERROR_MASK_DIR = f"/kaggle/working/pipeline/work/{SCENE}/error_masks"
_antialiasing_arg = "--antialiasing" if CKPT_ANTIALIASING else ""

!python {GS_REPO_TRAIN} -s {SOURCE_DIR} -m {MODEL_DIR} \
    --refine_from_iteration {CKPT_ITERATION} \
    --error_mask_dir {ERROR_MASK_DIR} \
    --iterations {REFINE_ITERATIONS} \
    --densify_until_iter 0 \
    --save_iterations {REFINE_ITERATIONS} \
    --test_iterations {REFINE_ITERATIONS} \
    --sh_degree {CKPT_SH_DEGREE} \
    {_antialiasing_arg}

## Bước 11 — Đo Score SAU khi tinh chỉnh, so với Bước 8

Nếu Score KHÔNG tăng (hoặc giảm), đợt tinh chỉnh này KHÔNG có lợi cho scene này — đừng
dùng checkpoint đã refine, giữ checkpoint gốc.

In [ ]:
_antialiasing_str = "on" if CKPT_ANTIALIASING else "off"  # tính trước, tránh biểu thức phức tạp trong dòng !
!python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {SCENE} \
    --poses_csv /kaggle/working/pipeline/work/{SCENE}/holdout/holdout_poses.csv \
    --out_dir /kaggle/working/pipeline/work/{SCENE}/holdout_renders \
    --iteration {REFINE_ITERATIONS} --sh_degree {CKPT_SH_DEGREE} \
    --antialiasing {_antialiasing_str}
!python /kaggle/working/pipeline/scripts/05_eval_metrics.py --scene {SCENE}

import csv

def _score_mean(path):
    rows = list(csv.DictReader(open(path)))
    return sum(float(r["score"]) for r in rows) / len(rows)

before = _score_mean(f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics_BEFORE_refine.txt")
after = _score_mean(f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics.txt")
print(f"Score TRƯỚC refine : {before:.4f}")
print(f"Score SAU refine   : {after:.4f}")
print(f"Chênh lệch         : {after - before:+.4f}")
if after > before:
    print("-> CÓ CẢI THIỆN — đáng cân nhắc dùng checkpoint đã refine.")
else:
    print("-> KHÔNG cải thiện (hoặc tệ hơn) — GIỮ checkpoint gốc, không dùng bản refine này.")

## Bước 12 (chỉ nếu Bước 11 cho thấy cải thiện) — Lưu checkpoint đã refine lên Drive

Giống hệt "Bước 7" của `kaggle_private.ipynb` — tải nguyên thư mục
`pipeline/work/<SCENE>/gs_model/` (giờ có thêm `point_cloud/iteration_{REFINE_ITERATIONS}/`)
lên Google Drive nếu muốn dùng cho bản nộp.

In [ ]:
print(f"Checkpoint đã refine (nếu quyết định dùng): "
      f"pipeline/work/{SCENE}/gs_model/point_cloud/iteration_{REFINE_ITERATIONS}/point_cloud.ply")
print("Bấm Save Version, vào tab Output, tải nguyên thư mục gs_model/ về rồi upload lên Drive "
      "(cùng cách làm với Bước 7 của kaggle_private.ipynb).")